In [12]:
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path
from urllib.parse import urlparse
from collections import Counter
from bs4 import BeautifulSoup

# =========================
# 1. PATHS
# =========================
JSON_PATH = Path("/Users/timvandegevel/Downloads/profiling-data/results_expanded.json")
HTML_DIR  = Path("/Users/timvandegevel/Downloads/profiling-data/html_front")
CSS_DIR   = Path("/Users/timvandegevel/Downloads/profiling-data/css_front")


# =========================
# 2. LOAD DATA
# =========================
df = pd.read_json(JSON_PATH, lines=True)

# Keep only the two thesis classes
df = df[df["mbfc-credibility-rating"].isin(["high credibility", "medium credibility"])].copy()

# Binary target
df["target"] = df["mbfc-credibility-rating"].map({
    "high credibility": 0,
    "medium credibility": 1
})


# =========================
# 3. MATCH URLS TO FILES
# =========================
html_files = set(os.listdir(HTML_DIR))
css_files = set(os.listdir(CSS_DIR))

def get_filename_from_url(url, extension):
    try:
        domain = urlparse(url).netloc
        return f"{domain}.{extension}"
    except:
        return None

df["html_filename"] = df["url"].apply(lambda x: get_filename_from_url(x, "html"))
df["css_filename"]  = df["url"].apply(lambda x: get_filename_from_url(x, "css"))

df["has_html"] = df["html_filename"].apply(lambda x: x in html_files)
df["has_css"]  = df["css_filename"].apply(lambda x: x in css_files)

# Final thesis sample: keep only rows with BOTH files
df["has_both_files"] = df["has_html"] & df["has_css"]
analysis_df = df[df["has_both_files"]].copy()

# Full paths
analysis_df["html_file_path"] = analysis_df["html_filename"].apply(lambda x: str(HTML_DIR / x))
analysis_df["css_file_path"]  = analysis_df["css_filename"].apply(lambda x: str(CSS_DIR / x))

print("Final sample shape:", analysis_df.shape)
print(analysis_df["mbfc-credibility-rating"].value_counts())

# Optional hard check
assert analysis_df.shape[0] == 3130, f"Expected 3130 rows, got {analysis_df.shape[0]}"
assert analysis_df["target"].value_counts().to_dict() == {0: 2561, 1: 569}, \
    f"Unexpected class counts: {analysis_df['target'].value_counts().to_dict()}"

Final sample shape: (3130, 44)
mbfc-credibility-rating
high credibility      2561
medium credibility     569
Name: count, dtype: int64


In [16]:
from functools import lru_cache

# =========================
# 4. HELPER FUNCTIONS
# =========================

CSS_COMMENT_RE = re.compile(r"/\*.*?\*/", re.S)
HTML_IN_CSS_RE = re.compile(r"<!DOCTYPE\s+html|<html\b|<head\b|<body\b|<script\b", re.I)
FONT_FACE_RE = re.compile(r"@font-face\b", re.I)
MEDIA_RE = re.compile(r"@media\b", re.I)
CLASS_RE = re.compile(r"\.(-?[_a-zA-Z]+[_a-zA-Z0-9-]*)")

BOOTSTRAP_PATTERNS = [
    re.compile(r"^(container|container-fluid|container-(sm|md|lg|xl|xxl))$"),
    re.compile(r"^row$"),
    re.compile(r"^col($|-)"),
    re.compile(r"^offset($|-)"),
    re.compile(r"^g([xy]?)-\d+$"),
    re.compile(r"^[mp][trblxy]?-\d+$"),
    re.compile(r"^btn($|-)"),
    re.compile(r"^d-"),
    re.compile(r"^justify-content-"),
    re.compile(r"^align-items-"),
    re.compile(r"^align-self-"),
    re.compile(r"^flex-"),
    re.compile(r"^order-"),
    re.compile(r"^shadow($|-)"),
    re.compile(r"^rounded($|-)"),
    re.compile(r"^position-"),
    re.compile(r"^(top|bottom|start|end)-"),
    re.compile(r"^translate-middle"),
    re.compile(r"^border($|-)"),
    re.compile(r"^w-\d+"),
    re.compile(r"^h-\d+"),
    re.compile(r"^bg-"),
]

WORDPRESS_PATTERNS = [
    re.compile(r"^wp-"),
    re.compile(r"^wp-block-"),
    re.compile(r"^has-"),
    re.compile(r"^is-"),
    re.compile(r"^align(left|right|center|wide|full)$"),
    re.compile(r"^align-"),
]

DRUPAL_PATTERNS = [
    re.compile(r"^text-align-"),
    re.compile(r"^align-(left|right|center|justify)$"),
    re.compile(r"^(visually-hidden|hidden|invisible|clearfix|nowrap|js-hide|js-show)$"),
    re.compile(r"^(fieldgroup|container-inline|system-status-|tablesort|item-list__)"),
]

ICON_PATTERNS = [
    re.compile(r"^fa($|-)"),
    re.compile(r"^fi-"),
    re.compile(r"^icon-"),
    re.compile(r"^bi($|-)"),
]

def strip_css_comments(text: str) -> str:
    return CSS_COMMENT_RE.sub("", text)

def extract_selectors(css_text: str):
    css_text = strip_css_comments(css_text)
    raw_heads = re.findall(r"([^{}]+)\{", css_text)

    selectors = []
    for head in raw_heads:
        head = head.strip()
        if not head or head.startswith("@"):
            continue

        parts = [p.strip() for p in head.split(",")]
        for part in parts:
            if part and len(part) < 300:
                selectors.append(part)

    return selectors

def extract_class_names_from_selectors(selectors):
    class_names = []
    for sel in selectors:
        class_names.extend(CLASS_RE.findall(sel))
    return class_names

def is_bootstrap_class(name: str) -> bool:
    return any(p.search(name) for p in BOOTSTRAP_PATTERNS)

def is_wordpress_class(name: str) -> bool:
    return any(p.search(name) for p in WORDPRESS_PATTERNS)

def is_drupal_class(name: str) -> bool:
    return any(p.search(name) for p in DRUPAL_PATTERNS)

def is_icon_class(name: str) -> bool:
    return any(p.search(name) for p in ICON_PATTERNS)

def is_boilerplate_class(name: str) -> bool:
    return (
        is_bootstrap_class(name)
        or is_wordpress_class(name)
        or is_drupal_class(name)
        or is_icon_class(name)
    )

def class_prefix(name: str) -> str:
    base = re.split(r"__|--", name)[0]
    parts = [p for p in re.split(r"[-_]", base) if p]
    return parts[0] if parts else base

def extract_html_classes(html_text: str):
    """
    Fast regex-based extraction of HTML class attributes.
    Much faster than BeautifulSoup for this use case.
    """
    class_values = re.findall(r'class\s*=\s*["\']([^"\']+)["\']', html_text, flags=re.I)
    html_classes = []

    for value in class_values:
        html_classes.extend(value.split())

    return html_classes

@lru_cache(maxsize=None)
def compute_css_features(css_path_str: str, html_path_str: str):
    default_result = {
        "css_file_exists": 0,
        "css_contains_html_flag": 0,
        "css_font_only_flag": 0,
        "css_rule_count": 0,
        "css_selector_count": 0,
        "css_unique_class_count": 0,
        "boilerplate_selector_share": 0.0,
        "wp_selector_share": 0.0,
        "bootstrap_selector_share": 0.0,
        "drupal_selector_share": 0.0,
        "icon_selector_share": 0.0,
        "custom_selector_share": 0.0,
        "custom_dominant_prefix_share": 0.0,
        "custom_top3_prefix_share": 0.0,
        "custom_prefix_family_count": 0,
        "custom_bem_share": 0.0,
        "media_query_density": 0.0,
        "html_css_class_coverage": 0.0,
        "unused_css_class_ratio": 0.0,
    }

    css_path = Path(css_path_str)
    html_path = Path(html_path_str)

    if not css_path.exists():
        return default_result

    try:
        css_text = css_path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return default_result

    selectors = extract_selectors(css_text)
    css_classes = extract_class_names_from_selectors(selectors)
    css_unique_classes = sorted(set(css_classes))

    css_rule_count = strip_css_comments(css_text).count("{")
    css_selector_count = len(selectors)
    css_unique_class_count = len(css_unique_classes)

    css_contains_html_flag = int(bool(HTML_IN_CSS_RE.search(css_text[:15000])))
    font_face_count = len(FONT_FACE_RE.findall(css_text))
    media_count = len(MEDIA_RE.findall(css_text))

    if css_unique_class_count == 0:
        wp_selector_share = 0.0
        bootstrap_selector_share = 0.0
        drupal_selector_share = 0.0
        icon_selector_share = 0.0
        boilerplate_selector_share = 0.0
        custom_selector_share = 0.0
        custom_dominant_prefix_share = 0.0
        custom_top3_prefix_share = 0.0
        custom_prefix_family_count = 0
        custom_bem_share = 0.0
    else:
        wp_count = sum(is_wordpress_class(c) for c in css_unique_classes)
        bootstrap_count = sum(is_bootstrap_class(c) for c in css_unique_classes)
        drupal_count = sum(is_drupal_class(c) for c in css_unique_classes)
        icon_count = sum(is_icon_class(c) for c in css_unique_classes)
        boilerplate_count = sum(is_boilerplate_class(c) for c in css_unique_classes)

        wp_selector_share = wp_count / css_unique_class_count
        bootstrap_selector_share = bootstrap_count / css_unique_class_count
        drupal_selector_share = drupal_count / css_unique_class_count
        icon_selector_share = icon_count / css_unique_class_count
        boilerplate_selector_share = boilerplate_count / css_unique_class_count
        custom_selector_share = 1 - boilerplate_selector_share

        custom_classes = [c for c in css_unique_classes if not is_boilerplate_class(c)]

        if len(custom_classes) == 0:
            custom_dominant_prefix_share = 0.0
            custom_top3_prefix_share = 0.0
            custom_prefix_family_count = 0
            custom_bem_share = 0.0
        else:
            prefix_counts = Counter(class_prefix(c) for c in custom_classes)
            prefix_values = sorted(prefix_counts.values(), reverse=True)

            custom_dominant_prefix_share = prefix_values[0] / len(custom_classes)
            custom_top3_prefix_share = sum(prefix_values[:3]) / len(custom_classes)
            custom_prefix_family_count = len(prefix_counts)
            custom_bem_share = sum(("__" in c or "--" in c) for c in custom_classes) / len(custom_classes)

    css_font_only_flag = int(
        (font_face_count >= 8) and (css_selector_count <= 25) and (css_unique_class_count <= 20)
    )

    media_query_density = media_count / max(css_rule_count, 1)

    # HTML-linked features
    if html_path.exists():
        try:
            html_text = html_path.read_text(encoding="utf-8", errors="ignore")
            html_classes = extract_html_classes(html_text)
            html_unique_classes = set(html_classes)

            if len(html_unique_classes) > 0:
                matched_classes = html_unique_classes.intersection(set(css_unique_classes))
                html_css_class_coverage = len(matched_classes) / len(html_unique_classes)
            else:
                html_css_class_coverage = 0.0

            if css_unique_class_count > 0:
                unused_css_classes = set(css_unique_classes) - html_unique_classes
                unused_css_class_ratio = len(unused_css_classes) / css_unique_class_count
            else:
                unused_css_class_ratio = 0.0

        except Exception:
            html_css_class_coverage = 0.0
            unused_css_class_ratio = 0.0
    else:
        html_css_class_coverage = 0.0
        unused_css_class_ratio = 0.0

    return {
        "css_file_exists": 1,
        "css_contains_html_flag": css_contains_html_flag,
        "css_font_only_flag": css_font_only_flag,
        "css_rule_count": css_rule_count,
        "css_selector_count": css_selector_count,
        "css_unique_class_count": css_unique_class_count,
        "boilerplate_selector_share": boilerplate_selector_share,
        "wp_selector_share": wp_selector_share,
        "bootstrap_selector_share": bootstrap_selector_share,
        "drupal_selector_share": drupal_selector_share,
        "icon_selector_share": icon_selector_share,
        "custom_selector_share": custom_selector_share,
        "custom_dominant_prefix_share": custom_dominant_prefix_share,
        "custom_top3_prefix_share": custom_top3_prefix_share,
        "custom_prefix_family_count": custom_prefix_family_count,
        "custom_bem_share": custom_bem_share,
        "media_query_density": media_query_density,
        "html_css_class_coverage": html_css_class_coverage,
        "unused_css_class_ratio": unused_css_class_ratio,
    }

In [17]:
# =========================
# QUICK TEST RUN (STRATIFIED SAMPLE)
# =========================

import time

# sample size: 100 high + 100 medium
test_df = pd.concat([
    analysis_df[analysis_df["target"] == 0].sample(n=100, random_state=42),
    analysis_df[analysis_df["target"] == 1].sample(n=100, random_state=42)
]).reset_index(drop=True)

print("Test sample shape:", test_df.shape)
print(test_df["target"].value_counts().to_string())

start_time = time.time()
feature_rows = []

total_rows = len(test_df)

for i, row in enumerate(test_df.itertuples(index=False), start=1):
    feats = compute_css_features(
        css_path_str=row.css_file_path,
        html_path_str=row.html_file_path
    )
    feature_rows.append(feats)

    if i % 25 == 0 or i == total_rows:
        elapsed = time.time() - start_time
        avg_per_row = elapsed / i
        remaining_rows = total_rows - i
        eta_seconds = avg_per_row * remaining_rows

        print(
            f"Processed {i}/{total_rows} rows "
            f"in {elapsed:.1f} seconds | "
            f"avg {avg_per_row:.3f}s/row | "
            f"ETA {eta_seconds:.1f} seconds"
        )

test_features_df = pd.DataFrame(feature_rows)

test_result_df = pd.concat(
    [test_df.reset_index(drop=True), test_features_df.reset_index(drop=True)],
    axis=1
)

print("\nPreview:")
display(test_result_df.head())

new_css_cols = test_features_df.columns.tolist()

print("\nMean by target:")
print(test_result_df.groupby("target")[new_css_cols].mean().to_string())

print("\nSaved quick test file.")
test_result_df.to_csv("css_new_features_test_sample.csv", index=False)

Test sample shape: (200, 44)
target
0    100
1    100
Processed 25/200 rows in 212.8 seconds | avg 8.511s/row | ETA 1489.4 seconds
Processed 50/200 rows in 443.5 seconds | avg 8.870s/row | ETA 1330.5 seconds
Processed 75/200 rows in 446.9 seconds | avg 5.959s/row | ETA 744.8 seconds
Processed 100/200 rows in 652.1 seconds | avg 6.521s/row | ETA 652.1 seconds
Processed 125/200 rows in 655.7 seconds | avg 5.245s/row | ETA 393.4 seconds
Processed 150/200 rows in 677.3 seconds | avg 4.516s/row | ETA 225.8 seconds
Processed 175/200 rows in 749.7 seconds | avg 4.284s/row | ETA 107.1 seconds
Processed 200/200 rows in 838.5 seconds | avg 4.193s/row | ETA 0.0 seconds

Preview:


,bias-rating,factual-reporting,country,mbfc’s-country-freedom-rating,media-type,traffic-popularity,mbfc-credibility-rating,url,label,source,...,drupal_selector_share,icon_selector_share,custom_selector_share,custom_dominant_prefix_share,custom_top3_prefix_share,custom_prefix_family_count,custom_bem_share,media_query_density,html_css_class_coverage,unused_css_class_ratio
0,right-center,high,japan,mostly free,newspaper,medium traffic,high credibility,https://japannews.yomiuri.co.jp/,right-center,https://mediabiasfactcheck.com/the-japan-news/,...,0.00000,0.00000,0.157333,0.203390,0.423729,29,0.016949,0.025404,0.000000,1.00000
1,left-center,high,usa,mostly free,radio station,medium traffic,high credibility,https://www.ctpublic.org/,left-center,https://mediabiasfactcheck.com/wnpr-connecticu...,...,0.00045,0.00045,0.994599,0.034389,0.088688,284,0.006335,0.088604,0.859316,0.89829
2,left-center,high,usa,mostly free,organization/foundation,medium traffic,high credibility,https://www.belfercenter.org/,left-center,https://mediabiasfactcheck.com/the-belfer-cent...,...,0.75000,0.00000,0.225000,0.444444,0.666667,6,0.111111,0.021739,0.005848,0.97500
3,least biased,mostly factual,usa,mostly free,magazine/website,medium traffic,high credibility,https://www.historynet.com/,least,https://mediabiasfactcheck.com/historynet-bias/,...,0.00000,0.00000,0.000000,0.000000,0.000000,0,0.000000,0.000000,0.000000,0.00000
4,left-center,high,united kingdom,NaN,newspaper,medium traffic,high credibility,https://bylinetimes.com/,left-center,https://mediabiasfactcheck.com/byline-times/,...,0.00000,0.00000,0.400000,0.250000,0.500000,7,0.000000,0.028986,0.016502,0.75000



Mean by target:
        css_file_exists  css_contains_html_flag  css_font_only_flag  css_rule_count  css_selector_count  css_unique_class_count  boilerplate_selector_share  wp_selector_share  bootstrap_selector_share  drupal_selector_share  icon_selector_share  custom_selector_share  custom_dominant_prefix_share  custom_top3_prefix_share  custom_prefix_family_count  custom_bem_share  media_query_density  html_css_class_coverage  unused_css_class_ratio
target                                                                                                                                                                                                                                                                                                                                                                                                                                                 
0                   1.0                    0.05                0.12         1306.99             1958.01

In [18]:
# =========================
# FULL RUN WITH CHECKPOINTS
# =========================

import time

start_time = time.time()
feature_rows = []

total_rows = len(analysis_df)

for i, row in enumerate(analysis_df.itertuples(index=False), start=1):
    feats = compute_css_features(
        css_path_str=row.css_file_path,
        html_path_str=row.html_file_path
    )
    feature_rows.append(feats)

    if i % 100 == 0 or i == total_rows:
        elapsed = time.time() - start_time
        avg_per_row = elapsed / i
        remaining_rows = total_rows - i
        eta_seconds = avg_per_row * remaining_rows

        print(
            f"Processed {i}/{total_rows} rows "
            f"in {elapsed:.1f} seconds | "
            f"avg {avg_per_row:.3f}s/row | "
            f"ETA {eta_seconds/60:.1f} minutes"
        )

        # save checkpoint
        checkpoint_df = pd.concat(
            [
                analysis_df.iloc[:i].reset_index(drop=True),
                pd.DataFrame(feature_rows).reset_index(drop=True)
            ],
            axis=1
        )
        checkpoint_df.to_csv("css_new_features_checkpoint.csv", index=False)
        print(f"Checkpoint saved at row {i}")

final_css_df = pd.concat(
    [analysis_df.reset_index(drop=True), pd.DataFrame(feature_rows).reset_index(drop=True)],
    axis=1
)

final_css_df.to_csv("css_new_features_results.csv", index=False)
print("\nSaved final file: css_new_features_results.csv")

Processed 100/3130 rows in 1785.0 seconds | avg 17.850s/row | ETA 901.4 minutes
Checkpoint saved at row 100
Processed 200/3130 rows in 3147.8 seconds | avg 15.739s/row | ETA 768.6 minutes
Checkpoint saved at row 200
Processed 300/3130 rows in 3670.2 seconds | avg 12.234s/row | ETA 577.0 minutes
Checkpoint saved at row 300
Processed 400/3130 rows in 5095.0 seconds | avg 12.737s/row | ETA 579.6 minutes
Checkpoint saved at row 400
Processed 500/3130 rows in 6099.3 seconds | avg 12.199s/row | ETA 534.7 minutes
Checkpoint saved at row 500
Processed 600/3130 rows in 6950.2 seconds | avg 11.584s/row | ETA 488.4 minutes
Checkpoint saved at row 600
Processed 700/3130 rows in 7788.7 seconds | avg 11.127s/row | ETA 450.6 minutes
Checkpoint saved at row 700
Processed 800/3130 rows in 8205.5 seconds | avg 10.257s/row | ETA 398.3 minutes
Checkpoint saved at row 800
Processed 900/3130 rows in 8554.0 seconds | avg 9.504s/row | ETA 353.2 minutes
Checkpoint saved at row 900
Processed 1000/3130 rows in 8

In [19]:
# =========================
# 6. INSPECT + SAVE OUTPUT
# =========================

new_css_cols = [
    "css_file_exists",
    "css_contains_html_flag",
    "css_font_only_flag",
    "css_rule_count",
    "css_selector_count",
    "css_unique_class_count",
    "boilerplate_selector_share",
    "wp_selector_share",
    "bootstrap_selector_share",
    "drupal_selector_share",
    "icon_selector_share",
    "custom_selector_share",
    "custom_dominant_prefix_share",
    "custom_top3_prefix_share",
    "custom_prefix_family_count",
    "custom_bem_share",
    "media_query_density",
    "html_css_class_coverage",
    "unused_css_class_ratio",
]

print("Missing values per feature:")
print(final_css_df[new_css_cols].isna().sum().to_string())

print("\nDescriptive statistics:")
print(final_css_df[new_css_cols].describe().to_string())

print("\nMean by target:")
print(final_css_df.groupby("target")[new_css_cols].mean().to_string())

print("\nNonzero share by target:")
nonzero_share = (final_css_df[new_css_cols] > 0).groupby(final_css_df["target"]).mean()
nonzero_share.index.name = "target"
print(nonzero_share.to_string())

# Save output
final_css_df.to_csv("css_new_features_results.csv", index=False)
print("\nSaved: css_new_features_results.csv")

# Optional quick checks
print("\nExamples flagged as css_contains_html_flag == 1:")
display(final_css_df.loc[final_css_df["css_contains_html_flag"] == 1, ["url", "css_filename", "target"]].head(20))

print("\nExamples flagged as css_font_only_flag == 1:")
display(final_css_df.loc[final_css_df["css_font_only_flag"] == 1, ["url", "css_filename", "target"]].head(20))

Missing values per feature:
css_file_exists                 0
css_contains_html_flag          0
css_font_only_flag              0
css_rule_count                  0
css_selector_count              0
css_unique_class_count          0
boilerplate_selector_share      0
wp_selector_share               0
bootstrap_selector_share        0
drupal_selector_share           0
icon_selector_share             0
custom_selector_share           0
custom_dominant_prefix_share    0
custom_top3_prefix_share        0
custom_prefix_family_count      0
custom_bem_share                0
media_query_density             0
html_css_class_coverage         0
unused_css_class_ratio          0

Descriptive statistics:
       css_file_exists  css_contains_html_flag  css_font_only_flag  css_rule_count  css_selector_count  css_unique_class_count  boilerplate_selector_share  wp_selector_share  bootstrap_selector_share  drupal_selector_share  icon_selector_share  custom_selector_share  custom_dominant_prefix_share  cus

,url,css_filename,target
26,https://www.thv11.com/,www.thv11.com.css,0
29,https://www.abc10.com/,www.abc10.com.css,0
35,https://www.kagstv.com/,www.kagstv.com.css,0
38,https://www.13newsnow.com/,www.13newsnow.com.css,0
53,https://pressfreedomtracker.us/,pressfreedomtracker.us.css,0
58,https://www.cbs19.tv/,www.cbs19.tv.css,0
64,https://vancouver.citynews.ca/,vancouver.citynews.ca.css,0
65,https://www.wfaa.com/,www.wfaa.com.css,0
68,https://www.wcnc.com/,www.wcnc.com.css,0
75,https://www.kare11.com/,www.kare11.com.css,0



Examples flagged as css_font_only_flag == 1:


,url,css_filename,target
6,https://www.federaltimes.com/,www.federaltimes.com.css,0
11,https://emersoncollegepolling.com/,emersoncollegepolling.com.css,0
41,https://www.chesterstandard.co.uk/,www.chesterstandard.co.uk.css,0
51,https://www.readingchronicle.co.uk/,www.readingchronicle.co.uk.css,0
56,https://www.hampshirechronicle.co.uk/,www.hampshirechronicle.co.uk.css,0
66,https://www.bromsgroveadvertiser.co.uk/,www.bromsgroveadvertiser.co.uk.css,0
72,https://www.bordercountiesadvertizer.co.uk/,www.bordercountiesadvertizer.co.uk.css,0
76,https://www.glasgowtimes.co.uk/,www.glasgowtimes.co.uk.css,0
90,https://www.maldonandburnhamstandard.co.uk/,www.maldonandburnhamstandard.co.uk.css,0
101,https://www.logically.ai/,www.logically.ai.css,0
